# K5 · Inference and submission (ensemble)
Settings: **Internet OFF** (required), accelerator **None** (CPU, no queue) or GPU.
Inputs: competition data, **K1** (shared code, decoders) and **every trained model** to use (K4, K4b, ...).
Then *Submit to competition*.

Every attached notebook output containing `model_final.pt` and `config.json` is loaded; the predictions are averaged.
Robustness: every study is guarded; on failure the mean prevalence is written instead of crashing.
Time guard: after 6 h only the first model is used, after 8 h the mean prevalence is written.

In [ ]:
import glob, os, sys, subprocess

def _walk_inputs(max_depth=5):
    """Walk /kaggle/input without ever entering the competition data (820,000 DICOM files)."""
    base = "/kaggle/input"
    for root, dirs, files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        dirs[:] = [d for d in dirs if d != "competitions" and "rsna-knee" not in d
                   and not d.endswith("_series") and depth < max_depth]
        yield root, dirs, files

def find(name):
    """Path of a file produced by another notebook attached as input (any mount depth)."""
    for root, dirs, files in _walk_inputs():
        if name in files:
            return os.path.join(root, name)
    p = f"/kaggle/working/{name}"
    return p if os.path.exists(p) else None

print("Attached inputs:")
for root, dirs, files in _walk_inputs(max_depth=3):
    if files:
        print("  ", root, "->", sorted(files)[:6])

wheels = [os.path.join(r, f) for r, d, fs in _walk_inputs() if os.path.basename(r) == "wheels" for f in fs if f.endswith(".whl")]
print(len(wheels), "wheels")
if wheels:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps"] + wheels, check=False)
common = find("knee_common.py")
if common is None:
    raise FileNotFoundError("knee_common.py not found: attach the K1 output (Add Input > Your Work > K1 series inventory).")
sys.path.insert(0, os.path.dirname(common))
print("shared code:", common)

sys.path.insert(0, os.path.dirname(find("knee_model.py")))
import knee_common as kc, knee_model as km
import json, time, numpy as np, pandas as pd, torch
from multiprocessing import Pool
T0 = time.time()
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGETS = kc.TARGETS
COMP = kc.find_comp_dir()
CFG = json.load(open(find("config.json")))
prior = json.load(open(find("prior.json"))) if find("prior.json") else {t: 0.2 for t in TARGETS}
PRIOR = np.array([prior[t] for t in TARGETS])

MODELS = []
for root, dirs, files in _walk_inputs():
    if "model_final.pt" in files and "config.json" in files:
        cfg = json.load(open(os.path.join(root, "config.json")))
        m = km.KneeMILModel(cfg["arch"], pretrained=False)
        m.load_state_dict(torch.load(os.path.join(root, "model_final.pt"), map_location="cpu"))
        MODELS.append((root, cfg, m.to(DEV).eval()))
MODELS.sort(key=lambda x: x[0])
assert MODELS, "No model found: attach the K4 training output."
assert len({cfg["size"] for _, cfg, _ in MODELS}) == 1, "All models must use the same volume size."
for root, cfg, _ in MODELS:
    print("model:", root, cfg["arch"], "seed", cfg.get("seed"))
torch.set_num_threads(max(1, os.cpu_count()))

test = pd.read_csv(f"{COMP}/test.csv")
tser = pd.read_csv(kc.series_csv_path(COMP, "test"))
SER_BY_STUDY = {st: g for st, g in tser.groupby("StudyInstanceUID")}
print(len(test), "test studies", DEV)

In [ ]:
def load_study(st):
    try:
        g = SER_BY_STUDY[st].copy()
        g["n"] = [len(os.listdir(kc.series_path(COMP, "test", st, s))) for s in g["SeriesInstanceUID"]]
        vol, mask, _ = kc.build_volume(COMP, "test", st, g, size=CFG["size"])
        return st, vol, mask, None
    except Exception as e:
        return st, None, None, f"{type(e).__name__}: {e}"

preds, fails = {}, []
studies = test["StudyInstanceUID"].tolist()
with Pool(os.cpu_count()) as pool:
    for k, (st, vol, mask, err) in enumerate(pool.imap(load_study, studies, chunksize=1)):
        p = None
        if err is None and time.time() - T0 < 8 * 3600:
            try:
                use = MODELS if time.time() - T0 < 6 * 3600 else MODELS[:1]
                outs = [km.predict_volume(m, vol, mask, DEV, kc.window_centers) for _, _, m in use]
                outs = [o for o in outs if o is not None and np.all(np.isfinite(o))]
                p = np.mean(outs, axis=0) if outs else None
            except Exception as e:
                err = f"{type(e).__name__}: {e}"
        if p is None or not np.all(np.isfinite(p)):
            fails.append((st, err)); p = PRIOR
        preds[st] = p
        if (k + 1) % 100 == 0:
            print(f"{k+1}/{len(studies)}  {(time.time()-T0)/60:.1f} min")
print("failures:", len(fails), fails[:5])

In [ ]:
sample = pd.read_csv(f"{COMP}/sample_submission.csv")
id_col, cols = sample.columns[0], list(sample.columns[1:])
sub = pd.DataFrame({id_col: studies})
P = np.stack([preds[s] for s in studies])
for j, c in enumerate(cols):
    k = TARGETS.index(c) if c in TARGETS else j      # by name, otherwise by position
    sub[c] = P[:, k]
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(sub.head()); print(sub.shape, f"{(time.time()-T0)/60:.1f} min in total")